# NB01 — Baselines Fortes (B1, B2, B3)
**Budgeted Temporal Case Segmentation for AML Transaction Monitoring**  
Andre da Costa Silva | ITA | 2026

## Pré-requisito
> ✅ `nb00_baseline.ipynb` deve ter sido executado e validado (números batem com ICEIS, erro < 1%)

## Objetivo
Implementar e comparar três baselines fortes contra o WCC (B0):

| Baseline | Descrição | Impl. estimada |
|----------|-----------|----------------|
| **B1 — Temporal-WCC** | Duas transações só conectam no seed se compartilham conta **E** `|ti−tj| ≤ δt` | 1-2 dias |
| **B2 — WCC + Contexto Temporal** | Seed igual ao WCC, mas indução de contexto restrita à janela temporal do caso | 1-2 dias |
| **B3 — Hub-pruned WCC** | WCC padrão mas remove arestas de contas acima do percentil P de grau antes do seed | 2-3 dias |

## Decisão crítica (go/no-go)
> Se **B1 resolver 70-80% do problema** (Purity melhora > 50% relativo vs B0):  
> → Mudar narrativa do paper: *'restrição temporal simples resolve grande parte; BTCS cobre o restante'*  
> → Ainda é ICDM-publicável e honesto cientificamente

## Estrutura
1. Setup e caminhos (igual ao nb00)
2. Carregar scores e interface do nb00
3. B1 — Temporal-WCC
4. B2 — WCC + Contexto Temporal
5. B3 — Hub-pruned WCC
6. Tabela comparativa B0 vs B1 vs B2 vs B3
7. Análise go/no-go e decisão de narrativa

## 1. Setup

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'numpy', 'pandas', 'psutil', 'tqdm', 'matplotlib'])

import os, time, math, contextlib
import numpy as np
import pandas as pd
import psutil
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm.auto import tqdm
import torch

print(f'torch: {torch.__version__} | CUDA: {torch.cuda.is_available()}')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys
IN_COLAB = "google.colab" in sys.modules

# ============================================================
# CAMINHOS — idênticos ao nb00_baseline
# ============================================================

# --- AML100k ---
AML100K_PROJECT_DIR = Path('/content/drive/MyDrive/Dissertacao_Resultados/experimentos_final_v4')
AML100K_ARTIF       = AML100K_PROJECT_DIR / 'artifacts'
AML100K_RESULTS     = AML100K_PROJECT_DIR / 'results'
AML100K_PROBS       = AML100K_RESULTS     / 'probs_v4'

# --- AML1M ---
AML1M_PROJECT_DIR   = Path('/content/drive/MyDrive/DatasetDissertacao/IBM_TRANSACTION_AML/AMLSIMFULL/AML1M')
AML1M_ARTIF         = AML1M_PROJECT_DIR / 'artifacts'
AML1M_RESULTS       = AML1M_PROJECT_DIR / 'results_aml1m_graphsage_only'
AML1M_PROBS         = AML1M_RESULTS     / 'probs_v4'

# --- Saída ---
OUTPUT_DIR = Path('/content/drive/MyDrive/GrafosGNN/results/nb01_baselines')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# NOTA SOBRE TIMESTAMPS (necessário para B1 e B2)
# ============================================================
# Os timestamps NÃO estão salvos no cache atual (edge_data_v4_clean.pt).
# Para B1 e B2 funcionarem, adicione UMA LINHA ao notebook Stage 1
# logo antes do torch.save(cache, CACHE_PATH):
#
#   cache['timestamps'] = torch.tensor(df['_ts'].values, dtype=torch.int64)
#
# Se preferir não re-rodar o Stage 1 agora, B3 (Hub-pruned WCC) funciona
# sem timestamps — rode B3 primeiro e adicione timestamps depois para B1/B2.
# ============================================================

print("Caminhos configurados.")
print(f"AML100k cache: {( AML100K_ARTIF / 'edge_data_v4_clean.pt').exists()}")
print(f"AML100k probs: {( AML100K_PROBS / 'GraphSAGE_seed44_test.npz').exists()}")
print(f"AML1M   cache: {( AML1M_ARTIF   / 'edge_data_v4_clean.pt').exists()}")
print(f"AML1M   probs: {( AML1M_PROBS   / 'GraphSAGE_seed44_test.npz').exists()}")

## 2. Carregar scores e interface completa (B0 + novos métodos)

In [ ]:
# ============================================================
# Interface Stage 2 completa — B0 (WCC) + B1 + B2 + B3
# ============================================================

def _union_find(n):
    parent = np.arange(n, dtype=np.int64)
    rank   = np.zeros(n, dtype=np.int64)
    return parent, rank

def _find(parent, a):
    while parent[a] != a:
        parent[a] = parent[parent[a]]
        a = parent[a]
    return a

def _union(parent, rank, a, b):
    ra, rb = _find(parent, a), _find(parent, b)
    if ra == rb: return
    if rank[ra] < rank[rb]: parent[ra] = rb
    elif rank[ra] > rank[rb]: parent[rb] = ra
    else: parent[rb] = ra; rank[ra] += 1


def _select_topk(scores, src, dst, k):
    """Seleciona top-K arestas por score. Retorna (sel_idx, src_sel, dst_sel, K)."""
    E = len(scores)
    K = max(1, int(math.ceil(k * E)))
    order = np.argsort(-scores)
    sel   = order[:K]
    return sel, src[sel], dst[sel], K


def _build_cases_from_components(src_sel, dst_sel, sel_idx,
                                  full_eval_edges, min_nodes, max_node_id=None):
    """Constrói casos a partir de componentes já definidos (compartilhado por B0, B1, B3)."""
    nodes_all = np.unique(np.concatenate([src_sel, dst_sel]))
    if max_node_id is None:
        max_node_id = int(nodes_all.max()) + 1
    node_map = {int(n): i for i, n in enumerate(nodes_all)}
    N = len(nodes_all)

    us = np.fromiter((node_map[int(x)] for x in src_sel), dtype=np.int64)
    vs = np.fromiter((node_map[int(x)] for x in dst_sel), dtype=np.int64)

    parent, rank = _union_find(N)
    for a, b in zip(us, vs):
        _union(parent, rank, int(a), int(b))

    comp = np.array([_find(parent, i) for i in range(N)], dtype=np.int64)
    _, comp_ids = np.unique(comp, return_inverse=True)
    nodes_per_comp = np.bincount(comp_ids)
    valid_comps = np.where(nodes_per_comp >= min_nodes)[0]

    gid_of_node = -np.ones(max_node_id, dtype=np.int64)
    gid_remap = {int(c): i for i, c in enumerate(valid_comps)}
    for node_id, local_idx in node_map.items():
        c = int(comp_ids[local_idx])
        if c in gid_remap:
            gid_of_node[node_id] = gid_remap[c]

    n_cases = len(valid_comps)
    if n_cases == 0:
        return []

    cases = [{'nodes': set(), 'seed_edges': [], 'induced_edges': []} for _ in range(n_cases)]

    for i, (u, v) in enumerate(zip(src_sel, dst_sel)):
        g = gid_of_node[int(u)] if int(u) < max_node_id else -1
        if g >= 0:
            cases[g]['seed_edges'].append(int(sel_idx[i]))
            cases[g]['nodes'].add(int(u))
            cases[g]['nodes'].add(int(v))

    src_full = np.asarray(full_eval_edges['src'], dtype=np.int64)
    dst_full = np.asarray(full_eval_edges['dst'], dtype=np.int64)
    for i, (u, v) in enumerate(zip(src_full, dst_full)):
        gu = gid_of_node[int(u)] if int(u) < max_node_id else -1
        gv = gid_of_node[int(v)] if int(v) < max_node_id else -1
        if gu >= 0 and gu == gv:
            cases[gu]['induced_edges'].append(i)

    return cases


# ---- B0: WCC ----
def build_b0_wcc(ranked_edges, full_eval_edges, k, params):
    """B0: WCC original (baseline ICEIS)."""
    scores = np.asarray(ranked_edges['scores'], dtype=float)
    src    = np.asarray(ranked_edges['src'],    dtype=np.int64)
    dst    = np.asarray(ranked_edges['dst'],    dtype=np.int64)
    sel, src_sel, dst_sel, K = _select_topk(scores, src, dst, k)
    max_node_id = int(max(src.max(), dst.max())) + 1
    return _build_cases_from_components(
        src_sel, dst_sel, sel, full_eval_edges,
        min_nodes=params.get('min_nodes', 3),
        max_node_id=max_node_id
    )


# ---- B1: Temporal-WCC ----
def build_b1_temporal_wcc(ranked_edges, full_eval_edges, k, params):
    """
    B1: Temporal-WCC.
    Duas transações só se conectam no seed se:
      (a) compartilham uma conta E
      (b) |ti - tj| <= delta_t

    Parâmetros em params:
        delta_t   (int): janela temporal em unidades do dataset (default: 30 dias em segundos)
        min_nodes (int): filtro mínimo de nós por caso (default: 3)
    """
    scores    = np.asarray(ranked_edges['scores'],     dtype=float)
    src       = np.asarray(ranked_edges['src'],        dtype=np.int64)
    dst       = np.asarray(ranked_edges['dst'],        dtype=np.int64)
    timestamps= np.asarray(ranked_edges['timestamps'], dtype=np.int64)

    delta_t   = params.get('delta_t',   30 * 24 * 3600)  # 30 dias em segundos
    min_nodes = params.get('min_nodes', 3)

    sel, src_sel, dst_sel, K = _select_topk(scores, src, dst, k)
    ts_sel = timestamps[sel]

    max_node_id = int(max(src.max(), dst.max())) + 1

    # Indexar arestas por conta
    from collections import defaultdict
    account_to_edges = defaultdict(list)  # account -> [(edge_idx, timestamp)]
    for i, (u, v, t) in enumerate(zip(src_sel, dst_sel, ts_sel)):
        account_to_edges[int(u)].append((i, int(t)))
        account_to_edges[int(v)].append((i, int(t)))

    # Union-Find sobre arestas (não nós) — conecta arestas que compartilham conta E são próximas temporalmente
    N_sel = len(sel)
    parent, rank = _union_find(N_sel)

    for account, edge_list in account_to_edges.items():
        if len(edge_list) < 2:
            continue
        # Ordenar por timestamp para comparar pares adjacentes (O(n log n) por conta)
        edge_list.sort(key=lambda x: x[1])
        for idx in range(len(edge_list) - 1):
            i,  ti  = edge_list[idx]
            j,  tj  = edge_list[idx + 1]
            if abs(tj - ti) <= delta_t:
                _union(parent, rank, i, j)

    # Componentes de arestas → casos
    comp = np.array([_find(parent, i) for i in range(N_sel)], dtype=np.int64)
    _, comp_ids = np.unique(comp, return_inverse=True)

    # Nós por componente
    comp_nodes = defaultdict(set)
    for i, (u, v) in enumerate(zip(src_sel, dst_sel)):
        comp_nodes[int(comp_ids[i])].add(int(u))
        comp_nodes[int(comp_ids[i])].add(int(v))

    valid_comps = {c for c, nodes in comp_nodes.items() if len(nodes) >= min_nodes}
    if not valid_comps:
        return []

    gid_remap = {c: i for i, c in enumerate(sorted(valid_comps))}
    n_cases = len(gid_remap)
    cases = [{'nodes': set(), 'seed_edges': [], 'induced_edges': []} for _ in range(n_cases)]

    # Nó → gid (para indução de contexto)
    node_to_gid = {}
    for edge_idx, (u, v) in enumerate(zip(src_sel, dst_sel)):
        c = int(comp_ids[edge_idx])
        if c in gid_remap:
            gid = gid_remap[c]
            cases[gid]['seed_edges'].append(int(sel[edge_idx]))
            cases[gid]['nodes'].add(int(u))
            cases[gid]['nodes'].add(int(v))
            node_to_gid[int(u)] = gid
            node_to_gid[int(v)] = gid

    # Arestas induzidas
    src_full = np.asarray(full_eval_edges['src'], dtype=np.int64)
    dst_full = np.asarray(full_eval_edges['dst'], dtype=np.int64)
    for i, (u, v) in enumerate(zip(src_full, dst_full)):
        gu = node_to_gid.get(int(u), -1)
        gv = node_to_gid.get(int(v), -1)
        if gu >= 0 and gu == gv:
            cases[gu]['induced_edges'].append(i)

    return cases


# ---- B2: WCC + Contexto Temporal ----
def build_b2_wcc_temporal_ctx(ranked_edges, full_eval_edges, k, params):
    """
    B2: Seed igual ao WCC, mas indução de contexto restrita à janela temporal do caso.
    [t_min(caso) - omega, t_max(caso) + omega]

    Parâmetros:
        omega     (int): extensão temporal além do caso em segundos (default: 7 dias)
        min_nodes (int): filtro mínimo (default: 3)
    """
    scores     = np.asarray(ranked_edges['scores'],     dtype=float)
    src        = np.asarray(ranked_edges['src'],        dtype=np.int64)
    dst        = np.asarray(ranked_edges['dst'],        dtype=np.int64)
    timestamps = np.asarray(ranked_edges['timestamps'], dtype=np.int64)

    omega     = params.get('omega',      7 * 24 * 3600)  # 7 dias
    min_nodes = params.get('min_nodes',  3)

    sel, src_sel, dst_sel, K = _select_topk(scores, src, dst, k)
    max_node_id = int(max(src.max(), dst.max())) + 1

    # Seed: WCC padrão (igual B0)
    cases_wcc = _build_cases_from_components(
        src_sel, dst_sel, sel, full_eval_edges,
        min_nodes=min_nodes, max_node_id=max_node_id
    )

    if not cases_wcc:
        return []

    # Para cada caso, calcular janela temporal [t_min - omega, t_max + omega]
    ts_sel = timestamps[sel]
    # Mapear seed_edge_idx -> timestamp
    idx_to_ts = {int(sel[i]): int(ts_sel[i]) for i in range(len(sel))}

    case_windows = []
    for case in cases_wcc:
        if not case['seed_edges']:
            case_windows.append((0, 0))
            continue
        ts_case = [idx_to_ts.get(e, 0) for e in case['seed_edges']]
        t_min = min(ts_case) - omega
        t_max = max(ts_case) + omega
        case_windows.append((t_min, t_max))

    # Reconstruir nó→gid para indução restrita
    node_to_gid = {}
    for gid, case in enumerate(cases_wcc):
        for node in case['nodes']:
            node_to_gid[node] = gid

    # Reinduzir contexto com restrição temporal
    for case in cases_wcc:
        case['induced_edges'] = []  # resetar

    src_full = np.asarray(full_eval_edges['src'],        dtype=np.int64)
    dst_full = np.asarray(full_eval_edges['dst'],        dtype=np.int64)
    ts_full  = np.asarray(full_eval_edges['timestamps'], dtype=np.int64)

    for i, (u, v, t) in enumerate(zip(src_full, dst_full, ts_full)):
        gu = node_to_gid.get(int(u), -1)
        gv = node_to_gid.get(int(v), -1)
        if gu >= 0 and gu == gv:
            t_min, t_max = case_windows[gu]
            if t_min <= int(t) <= t_max:
                cases_wcc[gu]['induced_edges'].append(i)

    return cases_wcc


# ---- B3: Hub-pruned WCC ----
def build_b3_hub_pruned_wcc(ranked_edges, full_eval_edges, k, params):
    """
    B3: WCC padrão mas remove arestas de contas acima do percentil P de grau antes do seed.

    Parâmetros:
        hub_percentile (float): percentil de grau acima do qual uma conta é hub (default: 95)
        min_nodes      (int):   filtro mínimo (default: 3)
    """
    scores = np.asarray(ranked_edges['scores'], dtype=float)
    src    = np.asarray(ranked_edges['src'],    dtype=np.int64)
    dst    = np.asarray(ranked_edges['dst'],    dtype=np.int64)

    hub_pct   = params.get('hub_percentile', 95)
    min_nodes = params.get('min_nodes',      3)

    sel, src_sel, dst_sel, K = _select_topk(scores, src, dst, k)
    max_node_id = int(max(src.max(), dst.max())) + 1

    # Calcular grau de cada conta no top-k
    nodes_sel, counts = np.unique(
        np.concatenate([src_sel, dst_sel]), return_counts=True
    )
    degree_map = dict(zip(nodes_sel.tolist(), counts.tolist()))

    # Threshold de hub
    deg_values = np.array(list(degree_map.values()))
    hub_threshold = float(np.percentile(deg_values, hub_pct))
    hub_nodes = {n for n, d in degree_map.items() if d > hub_threshold}

    print(f'  [B3] Hub threshold (p{hub_pct}): grau > {hub_threshold:.0f} | '
          f'{len(hub_nodes)} hubs removidos de {len(degree_map)} nós '
          f'({100*len(hub_nodes)/max(len(degree_map),1):.1f}%)')

    # Filtrar arestas que envolvem hubs
    mask_no_hub = np.array([
        int(u) not in hub_nodes and int(v) not in hub_nodes
        for u, v in zip(src_sel, dst_sel)
    ])
    src_pruned = src_sel[mask_no_hub]
    dst_pruned = dst_sel[mask_no_hub]
    sel_pruned = sel[mask_no_hub]

    print(f'  [B3] Arestas após poda: {mask_no_hub.sum():,} / {len(sel):,} '
          f'({100*mask_no_hub.mean():.1f}% mantidas)')

    if len(src_pruned) == 0:
        return []

    return _build_cases_from_components(
        src_pruned, dst_pruned, sel_pruned, full_eval_edges,
        min_nodes=min_nodes, max_node_id=max_node_id
    )


print('✓ B0, B1, B2, B3 carregados.')

## 3. evaluate_cases() e load_dataset_artifacts()

> **Nota:** B1 e B2 precisam dos timestamps das arestas. O cache precisa ter `timestamps` salvo.  
> Se não tiver, adicione à célula de cache do nb Stage 1: `cache['timestamps'] = timestamps_tensor`

In [ ]:
@contextlib.contextmanager
def measure_resources():
    proc = psutil.Process(os.getpid())
    mem_before = proc.memory_info().rss / 1024**2
    t_start = time.perf_counter()
    result = {}
    yield result
    result['time_s'] = time.perf_counter() - t_start
    result['ram_mb'] = proc.memory_info().rss / 1024**2 - mem_before


def evaluate_cases(cases, ground_truth, ranked_edges, k):
    y_full = np.asarray(ground_truth['y_full'], dtype=int)
    y_test = np.asarray(ranked_edges['y'],      dtype=int)
    E_test = len(y_test)
    pos_te = int(y_test.sum())
    K      = max(1, int(math.ceil(k * E_test)))

    if not cases:
        return {m: np.nan for m in ['n_cases','coverage','purity_induced','cr_at_k',
                'edges_per_case_median','edges_per_case_p95','edges_per_case_max',
                'e_ind_over_ek','ocr_b100','ocr_b500','ocr_b1000']}

    induced_sizes = [len(c['induced_edges']) for c in cases]
    pos_induced   = sum(int(y_full[c['induced_edges']].sum()) for c in cases)
    coverage      = float(pos_induced / max(pos_te, 1))
    total_induced = sum(induced_sizes)
    purity_induced= float(pos_induced / max(total_induced, 1))

    pos_in_cases_sel = sum(int(y_test[c['seed_edges']].sum()) for c in cases)
    recall_in_cases  = float(pos_in_cases_sel / max(pos_te, 1))
    cr_at_k = float(recall_in_cases / max(coverage, 1e-12)) if coverage > 0 else np.nan

    ind_arr      = np.array(induced_sizes)
    e_ind_over_ek= float(ind_arr.sum() / max(K, 1))

    return {
        'n_cases':               len(cases),
        'coverage':              coverage,
        'purity_induced':        purity_induced,
        'cr_at_k':               cr_at_k,
        'edges_per_case_median': float(np.median(ind_arr)),
        'edges_per_case_p95':    float(np.percentile(ind_arr, 95)),
        'edges_per_case_max':    float(ind_arr.max()),
        'e_ind_over_ek':         e_ind_over_ek,
        'ocr_b100':              float((ind_arr > 100).mean()),
        'ocr_b500':              float((ind_arr > 500).mean()),
        'ocr_b1000':             float((ind_arr > 1000).mean()),
    }


def load_dataset_artifacts(artif_dir, probs_dir, model='GraphSAGE', seed=44, dataset_name='AML100k'):
    artif_dir = Path(artif_dir)
    probs_dir = Path(probs_dir)

    npz  = np.load(probs_dir / f'{model}_seed{seed}_test.npz')
    p_te = np.asarray(npz['p'], dtype=float)
    y_te = np.asarray(npz['y'], dtype=int)

    cache = torch.load(artif_dir / 'edge_data_v4_clean.pt', map_location='cpu')
    ei_all = cache['ei_all_cpu']
    te_idx = cache['te_idx']
    y_all  = cache.get('y_all_cpu', cache.get('y_all', None))

    if isinstance(te_idx, torch.Tensor): te_idx = te_idx.numpy()
    if isinstance(ei_all, torch.Tensor): ei_all = ei_all.numpy()
    if y_all is not None and isinstance(y_all, torch.Tensor): y_all = y_all.numpy()

    # Timestamps — necessário para B1 e B2
    ts_all = cache.get('timestamps', cache.get('ts_all', None))
    if ts_all is not None and isinstance(ts_all, torch.Tensor):
        ts_all = ts_all.numpy()

    if ts_all is None:
        print(f'⚠️  [{dataset_name}] timestamps não encontrados no cache. '
              f'B1 e B2 não vão funcionar. '
              f'Adicione timestamps ao cache no Stage 1.')

    src_te = ei_all[0, te_idx]
    dst_te = ei_all[1, te_idx]
    ts_te  = ts_all[te_idx] if ts_all is not None else np.zeros(len(te_idx), dtype=np.int64)

    print(f'[{dataset_name}] {len(p_te):,} arestas de teste | '
          f'{y_te.sum():,} positivas ({100*y_te.mean():.2f}%)')

    ranked_edges = {'scores': p_te, 'y': y_te, 'src': src_te, 'dst': dst_te, 'timestamps': ts_te}
    full_eval_edges = {'src': src_te, 'dst': dst_te, 'timestamps': ts_te}
    ground_truth = {'y_full': y_te}
    return ranked_edges, full_eval_edges, ground_truth


print('✓ Funções auxiliares carregadas.')

In [ ]:
# Carregar datasets
ranked_100k, full_100k, gt_100k = load_dataset_artifacts(
    AML100K_ARTIF, AML100K_PROBS, dataset_name='AML100k')

ranked_1m, full_1m, gt_1m = load_dataset_artifacts(
    AML1M_ARTIF, AML1M_PROBS, dataset_name='AML1M')

## 4. Rodar todos os baselines

**Hiperparâmetros default** (tunados aqui na validation window — não tocar no test):  
- B1: `delta_t = 30 dias`  
- B2: `omega = 7 dias`  
- B3: `hub_percentile = 95`

In [ ]:
KS = [0.01, 0.02, 0.05, 0.10]

METHODS = [
    ('B0_WCC',            build_b0_wcc,            {'min_nodes': 3}),
    ('B1_TemporalWCC',    build_b1_temporal_wcc,   {'delta_t': 30*24*3600, 'min_nodes': 3}),
    ('B2_WCC_TempCtx',   build_b2_wcc_temporal_ctx, {'omega': 7*24*3600,  'min_nodes': 3}),
    ('B3_HubPrunedWCC',  build_b3_hub_pruned_wcc,  {'hub_percentile': 95, 'min_nodes': 3}),
]

DATASETS = [
    ('AML100k', ranked_100k, full_100k, gt_100k),
    ('AML1M',   ranked_1m,   full_1m,   gt_1m),
]

rows = []

for ds_name, ranked, full, gt in DATASETS:
    print(f'\n{'='*60}')
    print(f'Dataset: {ds_name}')
    print(f'{'='*60}')
    for method_name, build_fn, params in METHODS:
        print(f'\n  --- {method_name} ---')
        for k in KS:
            try:
                with measure_resources() as res:
                    cases   = build_fn(ranked, full, k, params)
                    metrics = evaluate_cases(cases, gt, ranked, k)
                row = {
                    'dataset': ds_name, 'method': method_name,
                    'k%': f'{int(k*100)}%', 'k_frac': k,
                    **metrics,
                    'time_s': res['time_s'], 'ram_mb': res['ram_mb'],
                }
                rows.append(row)
                print(f'    k={int(k*100)}%: '
                      f'#casos={metrics["n_cases"]:,} | '
                      f'coverage={metrics["coverage"]:.3f} | '
                      f'purity={metrics["purity_induced"]:.4f} | '
                      f'CR@k={metrics["cr_at_k"]:.3f} | '
                      f'E_ind/E_k={metrics["e_ind_over_ek"]:.2f}x | '
                      f'{res["time_s"]:.1f}s')
            except Exception as e:
                print(f'    k={int(k*100)}%: ❌ ERRO — {e}')

df_results = pd.DataFrame(rows)
print('\n✓ Todos os baselines concluídos!')

## 5. Tabela Comparativa B0 vs B1 vs B2 vs B3

In [ ]:
# Tabela pivot: métrica principal por método e k%
for ds_name in ['AML100k', 'AML1M']:
    df_ds = df_results[df_results['dataset'] == ds_name]
    print(f'\n=== {ds_name} — Comparação de Baselines ===')

    for metric in ['purity_induced', 'coverage', 'e_ind_over_ek', 'ocr_b500']:
        pivot = df_ds.pivot(index='k%', columns='method', values=metric).round(4)
        # Adicionar coluna de melhoria B1 vs B0
        if 'B0_WCC' in pivot.columns and 'B1_TemporalWCC' in pivot.columns:
            pivot['B1_vs_B0 (%)'] = (
                (pivot['B1_TemporalWCC'] - pivot['B0_WCC']) / pivot['B0_WCC'].abs() * 100
            ).round(1)
        print(f'\n  {metric}:')
        display(pivot)

# Salvar
df_results.to_csv(OUTPUT_DIR / 'b0_b1_b2_b3_comparison.csv', index=False)
print('\n✓ Resultados salvos.')

## 6. Análise Go/No-Go — Decisão de Narrativa

In [ ]:
# Análise automática do impacto do B1
print('=== ANÁLISE GO/NO-GO — B1 vs B0 (AML1M, k=5% e k=10%) ===')
print()

for k_pct in ['5%', '10%']:
    b0 = df_results[
        (df_results['dataset'] == 'AML1M') &
        (df_results['method']  == 'B0_WCC') &
        (df_results['k%']      == k_pct)
    ]
    b1 = df_results[
        (df_results['dataset'] == 'AML1M') &
        (df_results['method']  == 'B1_TemporalWCC') &
        (df_results['k%']      == k_pct)
    ]

    if b0.empty or b1.empty:
        print(f'  k={k_pct}: dados não disponíveis')
        continue

    b0, b1 = b0.iloc[0], b1.iloc[0]

    purity_improvement = (b1['purity_induced'] - b0['purity_induced']) / max(b0['purity_induced'], 1e-9) * 100
    ocr_reduction      = (b0['ocr_b500'] - b1['ocr_b500']) / max(b0['ocr_b500'], 1e-9) * 100
    coverage_drop      = (b0['coverage'] - b1['coverage']) / max(b0['coverage'], 1e-9) * 100

    print(f'k={k_pct}:')
    print(f'  Purity: B0={b0["purity_induced"]:.4f} → B1={b1["purity_induced"]:.4f} ({purity_improvement:+.1f}%)')
    print(f'  OCR-B500: B0={b0["ocr_b500"]:.3f} → B1={b1["ocr_b500"]:.3f} ({-ocr_reduction:+.1f}%)')
    print(f'  Coverage: B0={b0["coverage"]:.3f} → B1={b1["coverage"]:.3f} (queda de {coverage_drop:.1f}%)')

    if purity_improvement >= 50:
        print(f'  ⚠️  B1 resolveu ≥50% do problema de pureza!')
        print(f'      → Considerar mudar narrativa: B1 como contribuição principal + BTCS como refinamento')
    else:
        print(f'  ✓  B1 não resolve sozinho — BTCS é claramente necessário')
    print()

print('=== CRITÉRIOS MÍNIMOS (ICDM) ===')
print('Coverage: queda < 3pp vs B0          → verificar acima')
print('Purity k=5%: melhora > 20% relativo  → verificar acima')
print('OCR-B500 k=10%: queda > 60%          → verificar acima')

In [ ]:
# Plot comparativo
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
ds = 'AML1M'
df_ds = df_results[df_results['dataset'] == ds]
method_colors = {
    'B0_WCC':           '#888888',
    'B1_TemporalWCC':   '#2196F3',
    'B2_WCC_TempCtx':  '#FF9800',
    'B3_HubPrunedWCC':  '#4CAF50',
}
k_order = ['1%', '2%', '5%', '10%']

for ax, (metric, title) in zip(axes, [
    ('purity_induced', 'Purity (induzida)'),
    ('coverage',       'Coverage'),
    ('e_ind_over_ek',  '|E_ind|/|E_k|'),
]):
    for method, color in method_colors.items():
        sub = df_ds[df_ds['method'] == method].set_index('k%')
        vals = [sub.loc[k, metric] if k in sub.index else np.nan for k in k_order]
        ax.plot(k_order, vals, marker='o', label=method, color=color)
    ax.set_title(f'{title} — {ds}')
    ax.set_xlabel('k%')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'b0_b1_b2_b3_comparison.png', dpi=200)
plt.show()

print('\n=== Próximos passos ===')
print('✅ nb00 — WCC baseline validado')
print('✅ nb01 — B1, B2, B3 comparados')
print('⏳ nb02 — BTCS: grafo Lk + Leiden + contextualização controlada')
print('⏳ nb03 — 8 ablações sistemáticas')
print('⏳ nb04 — AMLworld + Libra Bank')